In [1]:
import random

import os

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.nn import functional as F


In [2]:
!pip install py7zr


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.5/75.5 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 493.4/493.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 6.8 MB/s eta 0:00:00


In [3]:
import py7zr


In [4]:
seed = 1723

random.seed(seed)
np.random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.use_deterministic_algorithms(mode=True)


In [5]:
## задача и роль, аудитория -> ритейл  

## загрузим датасет и зафикс сплит 
## out-of-time scheme 

## можете корр, добавить точки, 
## смешать выборки по оси t. 
## важен фикс для гипотез. 
## добавьте чек на lb 


In [6]:
for d, _, fl in os.walk('/kaggle/input'):
    for f in fl:
        with py7zr.SevenZipFile(os.path.join(d, f), mode='r') as z:
            z.extractall(path='/kaggle/working')


In [7]:
!ls /kaggle/working/


holidays_events.csv  oil.csv		    test.csv
items.csv	     sample_submission.csv  train.csv
__notebook__.ipynb   stores.csv		    transactions.csv


In [8]:
df = pd.read_csv(
    '/kaggle/working/train.csv', 
    skiprows=range(1, 66458909), 
    usecols=[1, 2, 3, 4], 
    parse_dates=['date'], 
    converters={
        'unit_sales': lambda u: np.log1p(float(u)) if float(u) > 0 else 0
    }
)

print (df.shape[0])
df.head()


59038132


,date,store_nbr,item_nbr,unit_sales
0,2016-01-01,25,105574,2.564949
1,2016-01-01,25,105575,2.302585
2,2016-01-01,25,105857,1.386294
3,2016-01-01,25,108634,1.386294
4,2016-01-01,25,108701,1.098612


In [9]:
maxi = (np.max(df['date']) - pd.DateOffset(days=16))

train_index = df[df['date'] <= maxi].index
valid_index = df[df['date'] > maxi].index

print (
    len(train_index), 
    len(valid_index)
)


57360788 1677344


In [10]:
## попробуйте реализовать схему с gap, зафикс метрики на нашем сплите 
## количество точек равно количеству моделей, 
## сейв пространства признаков создаёт разрыв 

## подсказка в https://www.kaggle.com/code/ceshine/lgbm-starter 

## (домашнее задание) 


In [11]:
## попробуйте реализовать схему с pl, зафикс и сравните метрики на нашем сплите 
## модель проходит шаг за шагом до посл точки в режиме авто-регрессии 

## https://www.kaggle.com/code/kneroma/m5-first-public-notebook-under-0-50  

## (домашнее задание) 


In [12]:
## посмотрим на возможности deep learning 
## схема с gap 


In [13]:
seq = torch.tensor([
    list(df[(df['store_nbr'] == 25) & (df['item_nbr'] == 105574)]['unit_sales'])[-32:-16], 
    list(df[(df['store_nbr'] == 25) & (df['item_nbr'] == 105575)]['unit_sales'])[-32:-16], 
    list(df[(df['store_nbr'] == 25) & (df['item_nbr'] == 105857)]['unit_sales'])[-32:-16] 
])

seq = seq.unsqueeze(2)

seq.shape


torch.Size([3, 16, 1])

In [14]:
class RNNCell(nn.Module):
    
    def __init__(self, 
                 input_size, 
                 hidden_size):
        super().__init__()
        self.w = nn.Linear(input_size, 
                           hidden_size) 
        self.u = nn.Linear(hidden_size, 
                           hidden_size) 

    def forward(self, x, h):
        return torch.tanh(self.w(x) + self.u(h))

class RNNLayer(nn.Module):
    
    def __init__(self, 
                 input_size, 
                 hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.cell = RNNCell(input_size, 
                            hidden_size)

    def forward(self, x):
        
        h = torch.zeros(x.size()[0], 
                        self.hidden_size)
        
        hs = []
        for t in range(x.size()[1]):
            h = self.cell(x[:, t, :], h)
            hs.append(h.unsqueeze(1))
        
        hs = torch.cat(hs, dim=1)  
        
        return hs, h


rnn = RNNLayer(1, 5)
hs, h = rnn(seq)

print (hs.shape, 
       h.shape)

h


torch.Size([3, 16, 5]) torch.Size([3, 5])


tensor([[-0.3470,  0.8701,  0.1593,  0.7679, -0.7162],
        [ 0.0381,  0.7519,  0.1687,  0.5424, -0.7113],
        [-0.3591,  0.5407,  0.4835,  0.3213, -0.1068]],
       grad_fn=<TanhBackward0>)

In [15]:
class MLPHead(nn.Module):
    
    def __init__(self, 
                 hidden_size, 
                 seq_length):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, seq_length)
        )

    def forward(self, x):
        return self.head(x)

mlp = MLPHead(5, 16)

mlp(h)


tensor([[-0.1031,  0.1195, -0.2665,  0.0246, -0.2479, -0.0151, -0.2137,  0.2273,
          0.1224, -0.1807, -0.2147,  0.0325, -0.1012,  0.0154,  0.2862, -0.0805],
        [-0.0729,  0.0032, -0.2554,  0.0988, -0.1413, -0.0342, -0.1787,  0.1382,
          0.0780, -0.1059, -0.1300,  0.0631, -0.1429,  0.0546,  0.3437, -0.0519],
        [-0.0638,  0.0165, -0.2444,  0.0994, -0.1705, -0.0245, -0.2109,  0.2144,
          0.1815, -0.2340, -0.1586, -0.0626, -0.1706,  0.0280,  0.3100,  0.0725]],
       grad_fn=<AddmmBackward0>)

In [16]:
## попробуйте реализовать схему с gap, зафикс и сравните метрики на нашем сплите  
## https://www.kaggle.com/code/cdeotte/gru-model-3rd-place-gold  

## попробуйте расширить пространство признаков 
## стремимся выйти на уровни grad boosting 

## (домашнее задание) 


In [17]:
## посмотрим на возможности deep learning 
## схема с pl 


In [18]:
encoder = RNNLayer(1, 5)
hs, h = encoder(seq)

print (hs.shape, 
       h.shape)

h


torch.Size([3, 16, 5]) torch.Size([3, 5])


tensor([[ 0.8447, -0.4931,  0.9744, -0.4385, -0.4544],
        [ 0.7425, -0.6672,  0.9401, -0.0564, -0.4874],
        [ 0.3795,  0.3966,  0.8770,  0.4447, -0.3302]],
       grad_fn=<TanhBackward0>)

In [19]:
class Seq2SeqDecoder(nn.Module):
    
    def __init__(self, 
                 hidden_size, 
                 seq_length):
        super().__init__()
        self.rnn_cell = RNNCell(1, hidden_size)
        self.mlp_head = MLPHead(hidden_size, 1)
        self.seq_length = seq_length

    def forward(self, h, y):
        
        ys = []
        for t in range(self.seq_length):
            h = self.rnn_cell(y, h)
            y = self.mlp_head(h)
            ys.append(y)
        
        ys = torch.cat(ys, dim=1)
        
        return ys

decoder = Seq2SeqDecoder(5, 16)

decoder(h, seq[:, -1, :])


tensor([[0.1169, 0.0402, 0.1262, 0.0684, 0.0872, 0.0860, 0.0819, 0.0851, 0.0838,
         0.0839, 0.0842, 0.0840, 0.0841, 0.0840, 0.0840, 0.0840],
        [0.1139, 0.0424, 0.1225, 0.0698, 0.0868, 0.0858, 0.0821, 0.0850, 0.0838,
         0.0840, 0.0841, 0.0840, 0.0841, 0.0840, 0.0840, 0.0840],
        [0.1084, 0.0571, 0.1049, 0.0778, 0.0838, 0.0857, 0.0829, 0.0844, 0.0840,
         0.0840, 0.0841, 0.0840, 0.0840, 0.0840, 0.0840, 0.0840]],
       grad_fn=<CatBackward0>)

In [20]:
## попробуйте реализовать схему с pl, зафикс и сравните метрики на нашем сплите  

## попробуйте расширить пространство признаков, 
## может отл для входа в энкодер и декодер 

## (домашнее задание) 


In [21]:
## энкодер возвращает тензоры (n, l, h) и (n, h), 
## посл стейт содержит сигнал о всей sequence 

## декодер принимает тензоры (n, d) и (n, h), 
## где d - размерность вектора с признаками, 
## а h - размерность хидден стейта 

## мы можем вернуться к тензору (n, l, h), 
## взвесить позиции по длине sequence 
## с помощью механизма кросс-внимания 

## контекст инд для каждого шага декодера, 
## выход на конкат с (n, d) 


In [22]:
encoder = RNNLayer(1, 5)
hs, h = encoder(seq)

print (hs.shape, 
       h.shape)

h


torch.Size([3, 16, 5]) torch.Size([3, 5])


tensor([[-0.8002, -0.9940,  0.9376, -0.6550, -0.1475],
        [-0.8482, -0.9908,  0.8970, -0.7059,  0.1435],
        [-0.4830, -0.9539,  0.5565, -0.5042,  0.2711]],
       grad_fn=<TanhBackward0>)

In [23]:
class DotAttention(nn.Module):
    
    def __init__(self, 
                 dropout_probability):
        super().__init__()
        self.dropout = nn.Dropout(dropout_probability)
    
    def forward(self, encoder_hs, decoder_h):

        decoder_h = decoder_h.unsqueeze(2)                   ## (n, h, 1)   
        
        attention_scores = torch.matmul(
            encoder_hs, 
            decoder_h
        )
        attention_scores = attention_scores.squeeze(2)       ## (n, l)   
        
        attention_ws = F.softmax(attention_scores, dim=-1)
        attention_ws = self.dropout(attention_ws)
        attention_ws = attention_ws.unsqueeze(1)             ## (n, 1, l)   
        
        context_vector = torch.matmul(
            attention_ws, 
            encoder_hs
        )                                                    ## (n, 1, h)   
        context_vector = context_vector.squeeze(1)           ## (n, h)   
        
        return context_vector

am = DotAttention(.20)

context_vector = am(
    hs, 
    h
)

context_vector


tensor([[-0.7183, -0.9409,  0.8013, -0.6099,  0.0525],
        [-1.0226, -1.2360,  1.1115, -0.8450, -0.0553],
        [-0.5615, -0.8878,  0.6184, -0.5000,  0.2030]],
       grad_fn=<SqueezeBackward1>)

In [24]:
class GeneralAttention(nn.Module):
    
    def __init__(self, 
                 hidden_size, 
                 dropout_probability):
        super().__init__()
        self.linear = nn.Linear(hidden_size, 
                                hidden_size, 
                                bias=False)
        self.dropout = nn.Dropout(dropout_probability)
    
    def forward(self, encoder_hs, decoder_h):

        encoder_proj = self.linear(encoder_hs)               ## (n, l, h)   

        decoder_h = decoder_h.unsqueeze(2)                   ## (n, h, 1)   
        
        attention_scores = torch.matmul(
            encoder_proj, 
            decoder_h
        )
        attention_scores = attention_scores.squeeze(2)       ## (n, l)   
        
        attention_ws = F.softmax(attention_scores, dim=-1)
        attention_ws = self.dropout(attention_ws)
        attention_ws = attention_ws.unsqueeze(1)             ## (n, 1, l)   
        
        context_vector = torch.matmul(
            attention_ws, 
            encoder_hs
        )                                                    ## (n, 1, h)   
        context_vector = context_vector.squeeze(1)           ## (n, h)   
        
        return context_vector

am = GeneralAttention(5, .20)

context_vector = am(
    hs, 
    h
)

context_vector


tensor([[-0.7827, -1.0645,  0.8763, -0.6679,  0.1007],
        [-0.7534, -0.9236,  0.8260, -0.6212, -0.0452],
        [-0.7318, -1.1350,  0.8163, -0.6460,  0.2201]],
       grad_fn=<SqueezeBackward1>)

In [25]:
class ConcatAttention(nn.Module):

    def __init__(self, 
                 hidden_size, 
                 dropout_probability):
        super().__init__()
        self.w = nn.Linear(hidden_size*2, hidden_size)
        self.v = nn.Linear(hidden_size, 1, bias=False)
        self.dropout = nn.Dropout(dropout_probability)

    def forward(self, encoder_hs, decoder_h):
        
        decoder_h = decoder_h.unsqueeze(1).expand(encoder_hs.size())   ## (n, l, h)   
        concat = torch.cat([encoder_hs, decoder_h], dim=-1)            ##  h -> 2h   

        concat_proj = torch.tanh(self.w(concat))                       ##  2h -> h  
        concat_proj = self.v(concat_proj)                              ## (n, l, 1)   
        
        concat_proj = concat_proj.squeeze(2)                           ## (n, l)   

        attention_ws = F.softmax(concat_proj, dim=-1)                  
        attention_ws = self.dropout(attention_ws)

        context_vector = torch.matmul(
            attention_ws.unsqueeze(1), 
            encoder_hs
        )
        context_vector = context_vector.squeeze(1)                     ## (n, h)   

        return context_vector

am = ConcatAttention(5, .20)

context_vector = am(
    hs, 
    h
)

context_vector


tensor([[-0.7265, -0.9929,  0.8206, -0.6166,  0.0719],
        [-0.9140, -1.1540,  0.9925, -0.7620,  0.0052],
        [-0.6047, -0.9768,  0.6759, -0.5421,  0.2095]],
       grad_fn=<SqueezeBackward1>)

In [26]:
class LuongSeq2SeqDecoder(nn.Module):
    
    def __init__(self, 
                 hidden_size, 
                 seq_length):
        super().__init__()
        self.rnn_cell = RNNCell(hidden_size+1, 
                                hidden_size)
        self.dot_am = DotAttention(.20)
        self.mlp_head = MLPHead(hidden_size, 
                                1)
        self.seq_length = seq_length

    def forward(self, hs, h, y):
        
        ys = []
        for t in range(self.seq_length):

            c = self.dot_am(hs, h)
            v = torch.cat([y, c], 
                          dim=-1)
            
            h = self.rnn_cell(v, h)
            y = self.mlp_head(h)
            ys.append(y)
        
        ys = torch.cat(ys, dim=1)
        
        return ys

decoder = LuongSeq2SeqDecoder(5, 16)

decoder(hs, h, seq[:, -1, :])


tensor([[-0.0387, -0.2025, -0.0547, -0.1031, -0.0755, -0.0725, -0.0731, -0.0764,
         -0.0710, -0.0644, -0.0752, -0.0814, -0.0937, -0.0890, -0.0826, -0.0759],
        [-0.0379, -0.1993, -0.0609, -0.0984, -0.0722, -0.0882, -0.0836, -0.0816,
         -0.0831, -0.0905, -0.0922, -0.0977, -0.0911, -0.0903, -0.0908, -0.0936],
        [-0.0164, -0.1583, -0.0572, -0.0938, -0.0665, -0.0810, -0.0810, -0.0810,
         -0.0864, -0.0807, -0.0697, -0.0686, -0.0737, -0.0738, -0.0771, -0.0780]],
       grad_fn=<CatBackward0>)

In [27]:
## https://www.kaggle.com/competitions/web-traffic-time-series-forecasting/writeups/arthur-suilin-1st-place-solution 

## советую посмотреть решение победителя турнира  
## инд -> единый context_vector для декодера,  
## интересная природа вектора с fingerprints 
